In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
            SparkSession
                .builder
                .master("local[*]")
                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark

In [ ]:
customer_df = spark.sql("select row_number() over(order by businessentityid) as customer_key,* from icebergtb.advworks_rdb.raw_dimcustomer")

In [ ]:
customer_df = (customer_df
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bln\b","Lane"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\brd\b","Road"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bDr\b","Drive"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bCt\b","Center"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bst\b","Street"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bblvd\b", "Boulevard"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bave\b","Avenue"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bpk\b","Parkway"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bterr\b", "Terrace"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bpl\b","Plaza"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bmt\b","Mountain"))
.withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bn.\b","North"))
)

customer_df= (customer_df
                .withColumn("addressnumber", regexp_extract("addressline1", r"(\d+)", 1))
                .withColumn("custaddress", regexp_replace("addressline1", r"\d+", ""))
            )


cleaned_customer = (customer_df 
                        .withColumn("custaddress", regexp_replace("custaddress", r"[^a-zA-Z ]", ""))

                    )

cleaned_customer = (cleaned_customer
                        .select('*').withColumn("custaddress", trim("custaddress"))
                    )

In [ ]:

cleaned_customer = (cleaned_customer
                        .select("*", "addressline2",
                            coalesce(cleaned_customer.addressline2, lit("Not Available"), "addressline2")   
                        .alias("complement_address"))
                    )                   

In [ ]:
cleaned_customer = (cleaned_customer.select("*",
                                    when(cleaned_customer.Gender == "M", "Male")
                                        .otherwise("Female").alias("customer_gender"),

                                    when(cleaned_customer.HomeOwnerFlag == 1, "yes")
                                        .otherwise("no").alias("ishomeowner"),

                                    when(col("maritalstatus") == "M", "Married")
                                        .otherwise("Single").alias("ismarried"),

                                    coalesce("numbercarsowned", lit("0").alias("iscarowned")).alias("iscarowner"))

)


In [ ]:
cleaned_customer = (cleaned_customer
                        .withColumn("age", year(current_date())- year(col("birthdate")))
)

cleaned_customer.printSchema()

In [ ]:
customer = cleaned_customer.select(
                            "customer_key",         "businessentityid",
                            "customerid",           "firstname",
                            "lastname",             "persontype",
                            "emailpromotion",       "typephone",
                            "phonenumber",          "postalcode",
                            "addresstype",          "emailaddress",
                            "territoryid",          "CountryRegionName",
                            "stateprovinceid",      "countryregioncode",
                            "stateprovincecode",    "statename",
                            "city",                 "custaddress",
                            "addressnumber",        "complement_address",
                            "TotalPurchaseYTD",     "DateFirstPurchase",
                            "BirthDate",            "age",
                            "ismarried",            "customer_gender",
                            "iscarowner",           "ishomeowner",
                            "TotalChildren",        "NumberChildrenAtHome",
                            "Education",            "Occupation",
                            "HomeOwnerFlag",        "NumberCarsOwned",
                            "inserted_date",        "modified_date",
                            "eltversion")